In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from tqdm.notebook import tqdm, trange
from torch import optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
train_data = np.load("ACAS_Xu_datasets/ACASXU_run2a_1_1_batch_2000_dataset/ACASXU_run2a_1_1_batch_2000_train.npz", allow_pickle=True)

test_data = np.load("ACAS_Xu_datasets/ACASXU_run2a_1_1_batch_2000_dataset/ACASXU_run2a_1_1_batch_2000_test.npz", allow_pickle=True)

X_train, X_val, Y_train, Y_val = train_test_split(
    train_data['X'], train_data['Y'],
    test_size=0.25,   # 0.25 of 0.8 = 0.2 of total data
    random_state=42
)

X_test = test_data['X']
Y_test = test_data['Y']

X_train = torch.tensor(X_train, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
Y_val = torch.tensor(Y_val, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, Y_train)
valid_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

batch_size = 2000

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False)

INPUT_SIZE = X_train.shape[1]
OUTPUT_SIZE = Y_train.shape[1]


In [3]:
# Calcola media e range (Max - Min) per ogni colonna (dim=0)
# Questo serve per scalare tutto tra circa -0.5 e 0.5
INPUT_MEAN = X_train.mean(dim=0)
INPUT_RANGE = X_train.max(dim=0).values - X_train.min(dim=0).values

# Evitiamo divisioni per zero (sicurezza)
INPUT_RANGE[INPUT_RANGE == 0] = 1.0

print("Media calcolata:", INPUT_MEAN)
print("Range calcolato:", INPUT_RANGE)

Media calcolata: tensor([ 3.0369e+04, -1.2216e-02,  3.0969e-03,  6.4991e+02,  5.9806e+02])
Range calcolato: tensor([6.0758e+04, 6.2828e+00, 6.2828e+00, 1.1000e+03, 1.2000e+03])


In [4]:
class ACASXuNet(torch.nn.Module):
    def __init__(self, input_size, output_size, means, ranges):
        super(ACASXuNet, self).__init__()

        self.register_buffer("means", means)
        self.register_buffer("ranges", ranges)

        self.layers = nn.Sequential(
            nn.Linear(input_size, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, output_size)
        )

    def forward(self, x):
        # 1. Normalizzazione interna: (x - mean) / range
        # ACAS Xu originale usa range per normalizzare gli input. 
        x_norm = (x - self.means) / self.ranges
        
        # 2. Forward pass nella rete
        return self.layers(x_norm)

In [5]:
class ACASXuNet(nn.Module):
    def __init__(self, input_size, output_size, means, ranges):
        super(ACASXuNet, self).__init__()
        
        # Registra le statistiche di normalizzazione come buffer (costanti, non addestrabili)
        self.register_buffer("means", means)
        self.register_buffer("ranges", ranges)

        # ModuleList allow better control in forward pass
        # ACAS Xu standard: 5 input -> 6 hidden layers (50 neuroni) -> 5 output
        self.hidden_layers = nn.ModuleList([
            nn.Linear(input_size, 50),
            nn.Linear(50, 50),
            nn.Linear(50, 50),
            nn.Linear(50, 50),
            nn.Linear(50, 50),
            nn.Linear(50, 50)
        ])

        # Output layer
        self.output_layer = nn.Linear(50, output_size)


    def forward(self, x, eps=0.0):
        """
        Args:
            x (Tensor): Input batch (N, 5).
            eps (float): Perturbation radius for IBP. Default 0.0 (Standard).
            
        Returns:
            If eps=0: Prediction tensor (N, 5).
            If eps>0: Tuple (center_predictions, lower_bound, upper_bound).
        """
        
        # 1. Input Normalization
        x_norm = (x - self.means) / self.ranges
        
        # --- Inference - Standard Training ---
        if eps == 0.0:
            out = x_norm
            for layer in self.hidden_layers:
                out = F.relu(layer(out)) # Linear -> ReLU
            out = self.output_layer(out) # Last linear layer
            return out

        # --- IBP Training ---
        
        # 1b. Epsilon Normalization
        # CRUCIAL: if real input varies by 'eps', normalized input varies by eps/range
        eps_norm = eps / self.ranges
        
        # Initial interval definition [l, u]
        lower_bound = x_norm - eps_norm
        upper_bound = x_norm + eps_norm
        
        # Propagation through hidden layers (Linear + ReLU)
        for layer in self.hidden_layers:
            lower_bound, upper_bound = self._propagate_linear(layer, lower_bound, upper_bound)
            lower_bound, upper_bound = F.relu(lower_bound), F.relu(upper_bound) # ReLU is monotonic: apply on min and max
            
        # Final layer propagation (Linear only)
        lower_bound, upper_bound = self._propagate_linear(self.output_layer, lower_bound, upper_bound)
        
        # Also compute center value to monitor pointwise accuracy
        center_pred = (lower_bound + upper_bound) / 2
        
        return center_pred, lower_bound, upper_bound

    def _propagate_linear(self, layer, lower_bound, upper_bound):
        """Internal helper: propagate intervals through Wx + b"""
        weight = layer.weight
        bias = layer.bias
        
        # Interval math:
        # Center (mu) and Radius (r) of input interval
        mu = (upper_bound + lower_bound) / 2
        r = (upper_bound - lower_bound) / 2
        
        # The center transforms linearly: W*mu + b
        mu_out = F.linear(mu, weight, bias)
        
        # The radius expands based on the ABSOLUTE value of the weights: |W|*r
        r_out = F.linear(r, torch.abs(weight))
        
        # Reconstruct the bounds [min, max]
        return mu_out - r_out, mu_out + r_out

In [6]:
def train_robust(model, iterator, optimizer, criterion, epsilon):

    model.train()
    epoch_loss = 0.0
    total_examples = 0

    for (x, y) in iterator:
        optimizer.zero_grad()
        
        if epsilon == 0.0:
            # Training Standard
            predictions = model(x)
            loss = criterion(predictions, y)
        else:
            # Training IBP (Robust)
            predictions, out_L, out_U = model(x, eps=epsilon)
            
            # Loss Standard
            loss_standard = criterion(predictions, y)
            
            # Robust Loss (Worst Case Error)
            # Evaluate the maximum distance between bounds (L, U) and target Y
            diff_L = (out_L - y)
            diff_U = (out_U - y)
            worst_case_error = torch.max(torch.abs(diff_L), torch.abs(diff_U))
            
            # MSE on Worst Case 
            loss_robust = torch.mean(worst_case_error ** 2)
            
            # Combination (Weighted Sum)
            # kappa=0.5 balances 50% accuracy and 50% robustness
            kappa = 0.5
            loss = (1 - kappa) * loss_standard + kappa * loss_robust

        loss.backward()
        optimizer.step()
        
        batch_size = x.size(0)
        total_examples += batch_size
        epoch_loss += loss.item() * batch_size

    return epoch_loss / total_examples

In [7]:
def validate_robust(model, iterator, criterion, epsilon):
    """
    Valuta il modello sul validation set calcolando sia l'MSE Standard
    che l'MSE Robusto (Worst Case Error).

    Args:
        epsilon: Il raggio di perturbazione da usare per il test di robustezza.
    """
    model.eval()
    
    total_standard_loss = 0.0
    total_robust_loss = 0.0
    total_examples = 0

    with torch.no_grad():
        for x, y in tqdm(iterator, desc="Validating", leave=False):
            
            # 1. Standard Accuracy 
            pred_standard = model(x)
            loss_standard = criterion(pred_standard, y)
            
            # 2. Robust Calculation (IBP)
            _, l, u = model(x, eps=epsilon)
            
            # Worst Case Error Calculation
            diff_L = (l - y)
            diff_U = (u - y)
            worst_case_diff = torch.max(torch.abs(diff_L), torch.abs(diff_U))
            loss_robust = torch.mean(worst_case_diff ** 2)

            # Aggregating statistics
            batch_size = x.size(0)
            total_standard_loss += loss_standard.item() * batch_size
            total_robust_loss += loss_robust.item() * batch_size
            total_examples += batch_size

    avg_standard_mse = total_standard_loss / total_examples
    avg_robust_mse = total_robust_loss / total_examples
    
    return avg_standard_mse, avg_robust_mse

In [8]:
def evaluate_model(model, test_loader, target_epsilon):
    """
    Executes final evaluation on the Test Set.
    Compares 'Standard' accuracy with 'Certified' robustness.
    """
    model.eval()

    # Stats 
    total_samples = 0
    total_std_loss = 0.0
    total_rob_loss = 0.0

    with torch.no_grad():
        for x, y in test_loader:
            batch_size = x.size(0)
            total_samples += batch_size

            # 1. Forward Pass IBP 
            pred, l, u = model(x, eps=target_epsilon)

            # 2. Calculate Standard MSE
            loss_std = torch.nn.functional.mse_loss(pred, y, reduction='sum')
            total_std_loss += loss_std.item()

            # 3. Calculate Robust MSE (Worst Case)
            diff_L = l - y
            diff_U = u - y
            worst_case_diff = torch.max(torch.abs(diff_L), torch.abs(diff_U))
            loss_rob = torch.sum(worst_case_diff ** 2)
            total_rob_loss += loss_rob.item()

    # Calculate averages
    avg_std_mse = total_std_loss / total_samples
    avg_rob_mse = total_rob_loss / total_samples


    results_data = {
        "Epsilon": target_epsilon,
        "Samples": total_samples,
        "MSE_Standard": avg_std_mse,
        "MSE_Robust": avg_rob_mse,
        "Certified_Safe": avg_rob_mse < 0.01 
    }
    
    # Returns a DataFrame with 1 row
    df = pd.DataFrame([results_data])
    return df

In [9]:
def epoch_time(start_time, end_time):
    """
    Calculates elapsed time between start and end timestamps.

    Args:
        start_time: float, start time in seconds
        end_time: float, end time in seconds

    Returns:
        elapsed_mins: integer number of minutes
        elapsed_secs: remaining seconds (after minutes are accounted for)
    """
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time // 60)
    elapsed_secs = int(elapsed_time % 60)
    return elapsed_mins, elapsed_secs

In [10]:
# Instantiate the TropMLP model with specified parameters
model = ACASXuNet(
    input_size=INPUT_SIZE,
    output_size=OUTPUT_SIZE,
    means=INPUT_MEAN,
    ranges=INPUT_RANGE
)

# Define Mean Squared Error as loss function for regression
criterion = nn.MSELoss()

# Use Adam optimizer with learning rate 0.03
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
EPOCHS = 50
WARMUP_EPOCHS = 5
MAX_EPSILON = 0.05      # Target finale di robustezza (es. 5% del range normalizzato)
RAMP_UP_EPOCHS = 30     # Epoche per passare da eps=0 a eps=MAX

best_robust_loss = float('inf')  # Tracciamo la loss robusta, non quella standard!
results_history = []             # Lista per il DataFrame finale

print(f"ACAS Xu Training (Target Eps: {MAX_EPSILON})")
print("-" * 60)

for epoch in trange(EPOCHS, desc="Epochs"):
    start_time = time.monotonic()

    # --- A. Calcolo Epsilon Corrente (Scheduler) ---
    if epoch < WARMUP_EPOCHS:
        current_eps = 0.0
    elif epoch < (WARMUP_EPOCHS + RAMP_UP_EPOCHS):
        # Crescita lineare
        progress = (epoch - WARMUP_EPOCHS) / RAMP_UP_EPOCHS
        current_eps = MAX_EPSILON * progress
    else:
        # Fase finale a massima robustezza
        current_eps = MAX_EPSILON

    # --- B. Training Step (Funzione 'train_robust' definita prima) ---
    train_loss = train_robust(model, train_loader, optimizer, criterion, epsilon=current_eps)

    # --- C. Validation Step (Funzione 'validate_robust' definita prima) ---
    # Usiamo lo stesso epsilon del training per monitorare se la rete "sta al passo"
    # Nota: validate_robust ora restituisce solo il worst-case MSE
    valid_robust_mse = validate_robust(model, valid_loader, criterion, epsilon=current_eps)
    
    # Calcoliamo anche l'MSE standard per confronto (opzionale, ma utile)
    valid_std_mse = validate_robust(model, valid_loader, criterion, epsilon=0.0)

    # --- D. Salvataggio del Modello Migliore (Basato su Robust MSE) ---
    
    #if epoch >= (WARMUP_EPOCHS + RAMP_UP_EPOCHS) and valid_robust_mse < best_robust_loss:
    #  best_robust_loss = valid_robust_mse
    #    torch.save(model.state_dict(), 'acas_xu_robust_best.pt')
    #    saved_msg = "-> Saved!"
    #else:
    #    saved_msg = ""
    

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    # --- E. Logging ---
    results_history.append({
        "Epoch": epoch + 1,
        "Epsilon": current_eps,
        "Train_Loss": train_loss,
        "Valid_MSE_Std": valid_std_mse,
        "Valid_MSE_Robust": valid_robust_mse,
        "Time_Sec": end_time - start_time
    })

    # Print statistica (compatta)
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\tValid Loss (MSE): {valid_std_mse:.3f}')
    print(f'\tValid Robust Loss (MSE): {valid_robust_mse:.3f}')

# --- 4. ANALISI FINALE ---
# Creazione DataFrame pandas dai risultati
df_results = pd.DataFrame(results_history)
print("\nTraining Completato.")
print(df_results.tail()) # Mostra le ultime epoche

# Se vuoi salvare la storia del training
# df_results.to_csv("training_log.csv", index=False)

ACAS Xu Training (Target Eps: 0.05)
------------------------------------------------------------


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/10 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 3)